# Tutorial 03: Running linear response calculations in PAOFLOW

This tutorial walks through on 'how to run' complete linear response functions that are implemented in PAOFLOW.

**What you will learn**

- How to actually run the calculation
- equations of linear response functions that are implemented

## Initialize PAOFLOW for linear response calculations:
- The linear response function, `pf.linear_response()`, requires following quantities:
    - PAO Hamiltonian (`pf.pao_hamiltonian()`,  and if needed,  `pf.interpolated_hamiltonian()`)
    - Eigenvalues and Eigenvectors (`pf.pao_eigh()`)
    - Gradient and Momenta of Hamiltonian (`pf.gradient_and_momenta()`)
    - Spin operators (`pf.spin_operator()`), if one calculates spin related response

**Linear responses can be called simply by specifying `response` as one of the string in following function: (See end of the tutorial for equation references)** </br></br>
`paoflow.linear_response(response = 'shc')`  ## *for spin Hall conductivity tensors* **(Evaluates: Eqn. 3)**</br></br>
`paoflow.linear_response(response = 'ree')` ## *for Rashba-Edelstein tensors* **(Evaluates: Eqn. 1)**</br></br>
`paoflow.linear_response(response = 'cond')` ## *for charge conductivity tensors* **(Evaluates: Eqn. 1)**</br></br>
`paoflow.linear_response(response = 'ahc')` ## *for anomolous Hall conductivity tensors* **(Evaluates: Eqn. 3)**</br></br>
**However, user can choose different variants of linear response (Eqn. 1 - 5), depending on the system. List of all possible arguments of function `linear_response()` is shown below. Refer wiki and equation references in section below to learn more about each arguments.**

In [ ]:
from PAOFLOW import PAOFLOW
if __name__== '__main__':
  paoflow = PAOFLOW.PAOFLOW(savedir='../pt.save/', ##.save directory
                            outputdir='output', 
                            verbose=1,
                            npool=8)
  data_controller = paoflow.data_controller
  arry,attr = data_controller.data_dicts()
  paoflow.read_atomic_proj_QE()
  paoflow.projectability()
  paoflow.pao_hamiltonian()
  # paoflow.interpolated_hamiltonian() ##usually require high mesh for convergence
  paoflow.pao_eigh()
  paoflow.gradient_and_momenta()
  paoflow.spin_operator()
  paoflow.linear_response(
      response = 'shc',
      gamma = 0.01,
      twoD = False,
      t_odd = False,
      full_chi2 = False,
      intraband = False,
      interband = False,
      s_tensor = None,
      a_tensor = None,
      eminH = -1.0,
      emaxH = 1.0,
      esize = 200)
paoflow.finish_execution()

# Wiki for `linear_response()` function:
**Use references below in conjunction with formulas given in next sections**
- `response` - *string*. *default* = '`shc`'. Possible strings:
  -  `ree` : Rashba-Edelstein effect
  -  `shc` : Spin Hall conductivity
  -  `ahc` : Anomolous Hall conductivity
  -  `cond` : charge conductivity </br></br>
- `t_odd` - *boolean*. *default* = '`False`'
  - Active only if `response` = '`ree`' or `response` = '`shc`'
  - If `t_odd` = `True`, $\chi^I$ (Eqn. 1) will be evaluated when `response` = '`shc`' and $\chi^{II}$ (Eqn. 3) if `response` = '`ree`'. By default (i.e. `t_odd` = `False`), setting `response` = '`shc`' evaluates $\chi^{II}$ while setting `response` = '`ree`' evaluates $\chi^I$. **Note:** It make sense to set `t_odd` = `True` only if your system breaks *time-reversible* symmetry, such as magnetic system.</br></br>
- `full_chi2` - *boolean*. *default* = '`False`'
  - Evaluates Eqn. (2) instead of Eqn. (3) for $\chi^{II}$ calculations if `full_chi2` = `True` </br></br>
- `intraband` - *boolean*. *default* = '`False`'. 
  - Evaluates Eqn. (4) instead of Eqn. (1) for $\chi^{I}$ calculations if `intraband` = `True` </br></br>
- `interband` - *boolean*. *default* = '`False`'.
  - Evaluates Eqn. (5) instead of Eqn. (1) for $\chi^{I}$ calculations if `interband` = `True` </br></br>
- `gamma` - *floats*. *default* = '`0.01`' eV
  - spectral broadening parameter</br></br>
- `s_tensor` - *nested list*. *default* = '`None`'
  - Tensors component of SHC to be evalauted
  - Ex: [[0,0,0],[2,1,3]]
  - first index: spin polarization direction
  - second item: spin current direction
  - third item: direction of applied electric field
  - By default, `shc_tensor = None`, evaluates all 27 components of SHC </br></br>
- `a_tensor` - *nested list*. *default* = '`None`'
  - Tensor components of REE, AHC, and charge conductivity to be evaluated
  - Ex: [[0,0],[2,1]]
  - first hand: direction of the reponse
  - second index: direction of applied electric field
  - By default, `s_tensor = None`, evaluates all 9 components of REE/AHC/conductivity </br></br>
- `eminH` - *floats*. *default* = '`-1.0`'
  - minimum energy for which linear response will be calculated </br></br>
- `emaxH` - *floats*. *default* = '`1.0`'
  - maximum energy for which linear response will be calculated </br></br>
- `esize` - *int*. *default* = '`200`'
  - number of energy steps between `eminH` and `emaxH`, for which responses will be calculated

# Linear response formulas implemented:
The linear response formalism implemented in PAOFLOW is *Kubo's linear response* formalism. The response that are in included in `linear_response()` function are response of the system under the action of applied field. In the constant relaxation time approximation, an observable $\delta \mathbf{A}$ induced in response to an external electric field $\mathbf{E}$ is expressed as:
$$
\delta A_i = \left(\chi^I_{ij}+\chi^{II}_{ij} \right)E_j
$$
where $\chi^{I}$ and $\chi^{II}$ are response tensors that is either *odd* (T-odd) or *even* (T-even) under the *time-reversible* symmetry operation. </br>
The response tensors are:
$$
\chi^{I}_{ij} = -\frac{e\hbar}{\pi}\sum_{k,n,m}\frac{\Gamma^2\text{Re}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{v}_j| \psi_{nk}\rangle  \right]  }
{\left[\left( E_F - E_{nk}\right)^2+\Gamma^2\right]  \left[\left( E_F - E_{mk}\right)^2+\Gamma^2\right]} \tag{1}
$$
and
$$
\chi^{II}_{ij} = 2e\hbar \sum_{k, n\neq m}^{\substack{\text{n occ}\\ \text{m unocc}}}
\frac
{\text{Im}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{v}_j|\psi_{nk}\rangle \right] \left(\Gamma^2 - (E_{nk}-E_{mk})^2\right)}
{\left[(E_{nk}-E_{mk})^2+\Gamma^2 \right]^2}
\tag{2}
$$
where $m$ and $n$ are band indices, $k$ is momenta, $E_F$ is Fermi energy or energy that one wish to compute above expressions, $\Gamma$ is spectral broadening, related to relaxation time constant $\tau$ as $\Gamma = \frac{\hbar}{2\tau}$, and $\hat{A}$ is operator for an observable $\mathbf{A}$ (see below for details), and $\hat{v}$ is the gradient operator. </br>

In the limit $\Gamma \rightarrow 0$, Eqn. (2) reduces to:
$$
\chi^{II}_{ij} \approx 2e\hbar \sum_{k, n\neq m}^{\substack{\text{n occ}\\ \text{m unocc}}}\frac
{\text{Im}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{v}_j|\psi_{nk}\rangle \right]}
{(E_{nk}-E_{mk})^2}
\tag{3}
$$

Further, for small $\Gamma$'s, Eqn.(1) can be decomposed into two parts:
$$
\chi^I_{\text{intra}} = -\frac{1}{V}\frac{e\hbar}{2\Gamma}\sum_{k,n} \text{Re}\left[\langle \psi_{nk}|\hat{A}_i|\psi_{nk}\rangle \langle \psi_{nk}|\hat{v}_j|\psi_{nk}\rangle \right]\delta E_{nk}
\tag{4}
$$
and 
$$
\chi^{I}_{\text{inter}} = -\frac{e\hbar}{V} \frac
{\text{Re} \left[\langle \psi_{nk}|\hat{A}_i|\psi_{mk}\rangle \langle \psi_{mk}|\hat{A}_i|\psi_{nk}\rangle \right] (E_{nk}-E_{mk})\Gamma}
{\left[(E_{nk}-E_{mk})^2+\Gamma^2\right]^2}
\tag{5}
$$

### Notes on operator $\hat{A}$
- $\hat{A} = \hat{v}$ </br>
      - $\chi^I$ - electrical conductivity tensor </br>
      - $\chi^{II}$ - Anomalous Hall effect (AHE) tensor</br></br>
- $\hat{A} = \hat{S}$  = spin operator</br>
        - $\chi^I$ - **T-even:** normal Rashba-Edelstein effect (REE) tensor; can exist in both magnetic and non-magnetic system</br>
      - $\chi^{II}$ - **T-odd:** magnetic Rashba-Edelstein effect (REE) tensor; vanishes for non-magnetic system</br> </br>
- $\hat{A} = \hat{j} = \frac{1}{2}\{\hat{S}_k, \hat{v}_j\}$ = spin current operator:</br>
      - $\chi^I$ - **T-odd:** magnetic spin Hall conductivity tensor; vanishes for non-magnetic system</br>
      - $\chi^{II}$ - **T-even:** normal spin Hall conductivity tensor; can exist in both magnetic and non-magnetic system

### References
- [https://doi.org/10.1038/s44306-025-00109-9](https://doi.org/10.1038/s44306-025-00109-9)
- [https://doi.org/10.1103/PhysRevB.90.174423](https://doi.org/10.1103/PhysRevB.90.174423)
- [https://doi.org/10.1103/PhysRevLett.126.127701](https://doi.org/10.1103/PhysRevLett.126.127701)
- [https://doi.org/10.1103/PhysRevLett.119.187204](https://doi.org/10.1103/PhysRevLett.119.187204)
- [https://doi.org/10.1038/s41467-024-51565-6](https://doi.org/10.1038/s41467-024-51565-6)
- [https://doi.org/10.1103/PhysRevB.91.134402](https://doi.org/10.1103/PhysRevB.91.134402)